In [1]:
print('yellou')

yellou


In [3]:
from pathlib import Path
import pandas as pd
import json

# 1. Rutas dinámicas usando pathlib (relativas a tu entorno de trabajo)
base_path = Path(r"C:\Users\Lenovo\Documents\scrapy_free_code_camp\fightmatrix\fightmatrix\spiders")
fighters_path = base_path / "ufcrosasbarcelos.json"
matchups_path = base_path / "rosasbarcelosnames.json"

# 2. Cargar los DataFrames
df_fighters = pd.read_json(fighters_path)
df_matchups = pd.read_json(matchups_path)

# 3. Separar los nombres y hacer el merge
df_matchups[["fighter_1", "fighter_2"]] = df_matchups["matchup"].str.split(" vs ", expand=True)

df_merged = df_matchups.merge(
    df_fighters.add_suffix("_f1"),
    left_on="fighter_1",
    right_on="fighter_f1",
    how="left",
).merge(
    df_fighters.add_suffix("_f2"),
    left_on="fighter_2",
    right_on="fighter_f2",
    how="left",
)

In [4]:
import ipywidgets as widgets
from IPython.display import display

def formatear_historial(historial):
    """Extrae un resumen legible usando saltos de línea HTML para que luzca ordenado en la celda."""
    if not isinstance(historial, list) or len(historial) == 0:
        return "No data"
    
    resumenes = []
    for pelea in historial:
        # Usamos <br> para que cada pelea salte de línea verticalmente
        resumen = f"<b>{pelea.get('result', '')}</b> {pelea.get('opponent', '')}</b> ({pelea.get('method', '')}) - R{pelea.get('round', '')} </b> ({pelea.get('time', '')})<br> {pelea.get('event', '')}</b> ({pelea.get('date', '')})"
        resumenes.append(resumen)
    return "<br>".join(resumenes)

def mostrar_comparativa(matchup_seleccionado):
    row = df_merged[df_merged['matchup'] == matchup_seleccionado].iloc[0]
    
    f1_name = row["fighter_1"]
    f2_name = row["fighter_2"]
    
    stats = [
        col.replace("_f1", "") 
        for col in df_merged.columns 
        if col.endswith("_f1") and col not in ["fighter_f1", "matchup_f1"]
    ]
    
    datos_comparativos = []
    for stat in stats:
        val_f1 = row.get(f"{stat}_f1", "N/A")
        val_f2 = row.get(f"{stat}_f2", "N/A")
        
        if stat == "sherdog_recent_fights":
            val_f1 = formatear_historial(val_f1)
            val_f2 = formatear_historial(val_f2)
        elif stat == "last_5" and isinstance(val_f1, list):
            val_f1 = "-".join(val_f1)
            val_f2 = "-".join(val_f2) if isinstance(val_f2, list) else val_f2

        datos_comparativos.append({
            "Estadística": stat.replace("sherdog_", "").replace("_", " ").title(),
            f1_name: val_f1,
            f2_name: val_f2
        })
        
    df_comp = pd.DataFrame(datos_comparativos).set_index("Estadística")
    
    # Esto le dice a Jupyter que expanda las celdas y permita leer texto con formato HTML (como los saltos de línea)
    display(df_comp.style.set_properties(**{
        'text-align': 'left',
        'white-space': 'normal',
        'max-width': '400px'
    }))
        
    df_comp = pd.DataFrame(datos_comparativos).set_index("Estadística")
    display(df_comp)

# Crear el menú desplegable con la lista de enfrentamientos
lista_matchups = df_merged['matchup'].dropna().unique().tolist()
dropdown = widgets.Dropdown(
    options=lista_matchups,
    description='Matchup:',
    disabled=False,
    layout={'width': 'max-content'}
)

# Conectar el menú a la función
widgets.interact(mostrar_comparativa, matchup_seleccionado=dropdown);

interactive(children=(Dropdown(description='Matchup:', layout=Layout(width='max-content'), options=('Raoni Bar…